Let's implement context steering from scratch!

The formula for the influence function is as follows:
$$F_{C,P}(x_i) = LLM(x_i
|C,P) − LLM(x_i
|∅,P) $$

where we seek to sample some next token $x_i$ based on comparing two distributions: one based on just the prompt $P$ with no additional context and one based on a prompt with additional context $C$.

In [1]:
import random 

# Let's first simulate the forward pass of our LLM. For a prompt with context and a prompt without context, 
# let's say we received the following logprobs:
logits = [round(random.uniform(-5, 0), 2) for _ in range(10)]
logits_no_context = [round(random.uniform(-5, 0), 2) for _ in range(10)]

print(f'Logits for LLM(l1): {logits_no_context}')
print(f'Logits for LLM(l1 + c): {logits}')

Logits for LLM(l1): [-3.29, -0.94, -2.56, -2.16, -2.33, -3.07, -2.94, -0.71, -0.62, -4.16]
Logits for LLM(l1 + c): [-1.24, -2.35, -0.32, -0.05, -0.85, -4.51, -2.87, -3.2, -3.77, -1.38]


In [2]:
import torch
import torch.nn.functional as F

# For numerically stability reasons, we take the log softmax of the logits
logprobs = F.log_softmax(torch.tensor(logits), dim=-1)
logprobs_no_context = F.log_softmax(torch.tensor(logits_no_context), dim=-1)

# We then calculate the influence as the difference between these logprobs
influence = logprobs - logprobs_no_context

influence

tensor([ 1.6166, -1.8434,  1.8066,  1.6766,  1.0466, -1.8734, -0.3634, -2.9234,
        -3.5834,  2.3466])

Next, let's calculate the subsequent probability distribution to sample from using context steering. To get the next token probabilities for some token $x$, we use the following calculation. 

$$ CoS_{\lambda}(x_i
|C,P) = LLM(x_i
|C,P) + \lambda · F_{C,P}(x_i)
 $$

Recall that we determine the level of contextual influence by tweaking lambda.

In [3]:
def apply_lambda(lmbda, logprobs, influence):
    # Apply the formula from above
    cos_distribution = logprobs + lmbda * influence

    # Normalize probabilities
    cos_distribution = F.log_softmax(cos_distribution, dim=-1)

    # If we want to work with probabilities instead of log probabilities, exponentiate
    cos_distribution = torch.exp(cos_distribution)

    return cos_distribution

Try out a few different values of influence:

In [4]:
for lmbda in [-1.0, 1.0, 3.0]:
    dist = apply_lambda(lmbda, logprobs, influence)
    print(f'Lambda: {lmbda}, Distribution: {dist.tolist()}')

Lambda: -1.0, Distribution: [0.020004315301775932, 0.20975667238235474, 0.04151057451963425, 0.06192648038268089, 0.052245210856199265, 0.024926917627453804, 0.028387470170855522, 0.2639997601509094, 0.2888617217540741, 0.008380841463804245]
Lambda: 1.0, Distribution: [0.09799626469612122, 0.0010150406742468476, 0.2973557114601135, 0.34204062819480896, 0.08185333758592606, 0.00011360015923855826, 0.0026509808376431465, 0.000147331491461955, 4.3063882912974805e-05, 0.17678405344486237]
Lambda: 3.0, Distribution: [0.05744485929608345, 5.877691933164897e-07, 0.2548881471157074, 0.2260657399892807, 0.015345532447099686, 6.195053714463938e-08, 2.9623888622154482e-05, 9.838842629505962e-09, 7.682308678091943e-10, 0.44622543454170227]


In practice, you can use the batched version of this code via the `apply_cos` function in `cos/core.py`.